# 无共享 token IDs 的跨语言 SVA：实验结果

主线：检验 original/clone 间的因果迁移；训练轨迹作为补充。本文的“跨语言”仅指英语及其克隆语言。

此 notebook 只读取 `reports/cross_language_sva/data/` 中的紧凑汇总，不加载模型或大型 activation 文件。所有图均可重新生成 PNG。完整原始数据的统计重算命令见报告目录的 README。

**统计口径**：固定 1,280 对 patching cohort，六 checkpoint 一致；行为评估为完整 3,200 对测试集。L8H3 是代码中的零起始索引。轨迹区间为逐点 95% CI，未做跨 checkpoint 多重比较校正，不能据此确定 circuit 的首次形成时刻。

In [ ]:
from pathlib import Path
import csv, json, hashlib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/cross_language_sva/data/manifest.json').is_file())
PACK = ROOT / 'reports/cross_language_sva'
DATA, FIGURES = PACK / 'data', PACK / 'figures'
FIGURES.mkdir(exist_ok=True)
manifest = json.loads((DATA / 'manifest.json').read_text())
for name, expected in manifest['compact_hashes'].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == expected, f'Stale or changed summary: {name}'

def load(name):
    with (DATA / name).open() as f:
        return list(csv.DictReader(f))

def select(rows, **conditions):
    return [r for r in rows if all(str(r[k]) == str(v) for k, v in conditions.items())]

def ordered(rows):
    return sorted(rows, key=lambda r: int(r['step']))

def numbers(rows, field):
    return np.array([float(r[field]) for r in rows])

def table(rows, columns):
    header = '| ' + ' | '.join(columns) + ' |'
    body = ['| ' + ' | '.join(str(r[c]) for c in columns) + ' |' for r in rows]
    display(Markdown('\n'.join([header, '|' + '|'.join(['---'] * len(columns)) + '|', *body])))

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 220, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
LANG_COLORS = {'original': '#2864a0', 'clone': '#c45135'}
DIRECTIONS = ['original_to_original', 'clone_to_clone', 'original_to_clone', 'clone_to_original']
CROSS = DIRECTIONS[2:]
LABELS = {d: d.replace('_to_', ' → ') for d in DIRECTIONS}
TASKS = ['simple', 'pp_attractor', 'object_relative', 'subject_relative']
TASK_LABELS = {'pooled_distractor': 'Pooled distractors', 'simple': 'Simple', 'pp_attractor': 'PP attractor',
               'object_relative': 'Object relative', 'subject_relative': 'Subject relative'}
NULLS = ['opposite-number', 'opposite-number-shuffled']
behavior = load('behavior_full_test.csv')
effects = load('trajectory_effects.csv')
changes = load('trajectory_changes_from_5k.csv')
raw = load('control_means.csv')
coverage = load('coverage.csv')

def save(fig, name):
    fig.savefig(FIGURES / f'{name}.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)

def curve(ax, rows, label, color, linestyle='-', ci='cluster'):
    rows = ordered(rows)
    x, y = numbers(rows, 'step') / 1000, numbers(rows, 'mean')
    ax.plot(x, y, marker='o', label=label, color=color, linestyle=linestyle)
    ax.fill_between(x, numbers(rows, ci + '_ci_low'), numbers(rows, ci + '_ci_high'), color=color, alpha=.12)
    ax.set_xlabel('Training step (thousands)')

assert len(manifest['checkpoints']) == 6
assert len({r['step'] for r in behavior}) == 6
assert len(effects) == 240
print(f"6 checkpoints; 96 patching archives; {manifest['iterations']:,} bootstrap replicates; seed 42")
table(coverage, ['task', 'test_pairs', 'patched_pairs', 'coverage', 'prompt_clusters'])

## 0. 预训练与语言模型质量

以下曲线来自训练日志；best checkpoint 由平均 validation loss 选择，不是按 SVA 或 patching 效果挑选。测试 PPL 来自完整 held-out test split。

In [ ]:
training = load('training_validation.csv')
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for language, color in LANG_COLORS.items():
    x = numbers(training, 'step') / 1000
    axes[0].plot(x, numbers(training, language + '_loss'), color=color, label=language.title())
    axes[1].plot(x, numbers(training, language + '_perplexity'), color=color, label=language.title())
axes[0].set(title='Validation loss', xlabel='Training step (thousands)', ylabel='Cross entropy')
axes[1].set(title='Validation perplexity', xlabel='Training step (thousands)', ylabel='Perplexity', yscale='log')
for ax in axes:
    ax.axvline(77.5, linestyle=':', color='gray', label='Best checkpoint')
    ax.legend()
save(fig, '00_training_validation')
ppl = [r for r in load('test_perplexity.csv') if r['model'] == 'trained']
table(ppl, ['language', 'average_cross_entropy', 'perplexity', 'valid_next_token_positions'])

## 1. 模型是否掌握 SVA？

使用完整测试集，每个 checkpoint 都评估相同的 3,200 对句子。Prompt accuracy 是 6,400 个提示上的正确率；pair accuracy 要求一对中的两种主语数都预测正确。阴影为按提示分组的配对 bootstrap 95% CI，不是多训练 seed 的误差条。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for ax, metric, title in zip(axes, ['accuracy', 'pair_accuracy'], ['Prompt accuracy', 'Pair accuracy']):
    for language in LANG_COLORS:
        curve(ax, select(behavior, task='overall', language=language, metric=metric), language.title(), LANG_COLORS[language])
    if metric == 'accuracy':
        ax.axhline(.5, color='gray', linestyle=':', label='50% reference')
    ax.set(title=title, ylabel='Accuracy', ylim=(0, 1))
    ax.legend()
save(fig, '01_behavior_full_test')
rows = []
for step in sorted({int(r['step']) for r in behavior}):
    row = {'step': step}
    for language in LANG_COLORS:
        for metric in ['accuracy', 'pair_accuracy']:
            r = select(behavior, step=step, language=language, metric=metric, task='overall')[0]
            row[f'{language} {metric}'] = f"{float(r['mean']):.2%}"
    rows.append(row)
table(rows, list(rows[0]))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True, layout='constrained')
for ax, task in zip(axes.flat, TASKS):
    for language in LANG_COLORS:
        curve(ax, select(behavior, task=task, language=language, metric='accuracy'), language.title(), LANG_COLORS[language])
    ax.axhline(.5, color='gray', linestyle=':')
    ax.set(title=TASK_LABELS[task], ylabel='Prompt accuracy', ylim=(.4, 1))
axes.flat[0].legend()
save(fig, '02_behavior_by_task')

## 2. Best checkpoint 的共享证据

比较 source clean patch 与两种 number 对照对 target 动词偏好的影响：

`effect = ΔLD(clean source patch) − ΔLD(number-control source patch)`。

这是原始 log-probability difference 的差值，不是准确率增量或 recovery 百分比。正值表示 clean source 相比对照更能推动 target 朝 source clean 主语数对应的动词预测。以下两种区间均基于同一批数据，不是两次独立复现。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout='constrained')
groups = ['pooled_distractor', 'pp_attractor', 'object_relative', 'subject_relative']
for ax, null in zip(axes, NULLS):
    for index, direction in enumerate(CROSS):
        rows = [select(effects, checkpoint='best', direction=direction, control=null, task=t)[0] for t in groups]
        y = np.arange(len(groups)) + (index - .5) * .22
        means = numbers(rows, 'mean')
        lo, hi = numbers(rows, 'cluster_ci_low'), numbers(rows, 'cluster_ci_high')
        ax.errorbar(means, y, xerr=[means-lo, hi-means], fmt='o', capsize=3,
                    color=list(LANG_COLORS.values())[index], label=LABELS[direction])
    ax.axvline(0, color='gray', linestyle=':')
    ax.set(title=f'vs {null}', xlabel='Clean-minus-control effect (raw ΔLD)', yticks=np.arange(len(groups)),
           yticklabels=[TASK_LABELS[t] for t in groups])
    ax.legend()
axes[0].invert_yaxis()
save(fig, '03_best_cross_language')
rows = []
for r in select(effects, checkpoint='best'):
    if r['direction'] in CROSS and r['task'] in groups:
        rows.append({'direction': r['direction'], 'null': r['control'], 'task': r['task'],
            'effect': f"{float(r['mean']):+.4f}",
            'row 95% CI': f"[{float(r['row_ci_low']):+.4f}, {float(r['row_ci_high']):+.4f}]",
            'prompt-cluster 95% CI': f"[{float(r['cluster_ci_low']):+.4f}, {float(r['cluster_ci_high']):+.4f}]", 'N': r['num_examples']})
table(rows, list(rows[0]))

## 3. L8H3 的训练轨迹

四个方向始终使用同一 cohort，没有在每个 checkpoint 重新筛选答对的句子。主图报告 raw ΔLD，避免早期 clean/corrupted 差值接近零导致归一化 recovery 不稳定。

以下是观察到的六个时间点；连线只用于阅读，不表示中间 checkpoint 已经测量。区间为探索性、逐点区间；跨越零表示证据不明确，不等于 circuit 不存在。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, layout='constrained')
for ax, null in zip(axes, NULLS):
    for direction, color in zip(CROSS, LANG_COLORS.values()):
        curve(ax, select(effects, task='pooled_distractor', direction=direction, control=null), LABELS[direction], color)
    ax.axhline(0, color='gray', linestyle=':')
    ax.set(title=f'L8H3 vs {null}', ylabel='Clean-minus-control effect (raw ΔLD)')
    ax.legend()
save(fig, '04_cross_language_trajectory')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True, layout='constrained')
for i, direction in enumerate(CROSS):
    for j, task in enumerate(TASKS[1:]):
        ax = axes[i, j]
        for null, color, style in zip(NULLS, ['#34699a', '#ad572e'], ['-', '--']):
            curve(ax, select(effects, task=task, direction=direction, control=null), null, color, style)
        ax.axhline(0, color='gray', linestyle=':')
        ax.set(title=f'{LABELS[direction]} | {TASK_LABELS[task]}', ylabel='Effect (raw ΔLD)')
axes[0, 0].legend(fontsize=8)
save(fig, '05_cross_language_by_task')

## 4. 对照是否正常？

同语言方向提供参照；same-number-shuffled 检查效果是否依赖同一词汇实例。不能把 same-number-shuffled 当成“应当为零”的负对照，因为它保留了数信息。

注意：跨语言 opposite-number patch 仍会替换语言来源，因此其 raw ΔLD 不保证为零；主结论使用 clean-minus-control 差值。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
for direction, color in zip(DIRECTIONS, ['#547f39', '#9c6dab', '#2864a0', '#c45135']):
    curve(axes[0], select(effects, task='pooled_distractor', direction=direction, control='opposite-number-shuffled'), LABELS[direction], color)
axes[0].axhline(0, color='gray', linestyle=':')
axes[0].set(title='Within- and cross-language transfer', ylabel='Clean-minus-control effect (raw ΔLD)')
axes[0].legend(fontsize=8)
controls = ['clean', 'opposite-number', 'same-number-shuffled', 'opposite-number-shuffled']
x = np.arange(4)
for index, control in enumerate(controls):
    means = [float(select(raw, checkpoint='best', direction=d, control=control, task='pooled_distractor')[0]['mean_delta_ld']) for d in DIRECTIONS]
    axes[1].bar(x + (index - 1.5)*.19, means, .19, label=control)
axes[1].axhline(0, color='gray', linestyle=':')
axes[1].set(xticks=x, xticklabels=['O→O', 'C→C', 'O→C', 'C→O'], ylabel='Mean raw ΔLD', title='Best checkpoint: source controls')
axes[1].legend(fontsize=8)
save(fig, '06_controls')

## 5. 与 5k 相比是否发生变化？

先逐样本计算“当前 checkpoint 效应 − 5k 效应”，再进行配对重采样；不通过两组独立误差条是否重叠来判断变化。这是事后探索性比较，不是对三阶段假说的预注册检验。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, layout='constrained')
for ax, null in zip(axes, NULLS):
    for direction, color in zip(CROSS, LANG_COLORS.values()):
        curve(ax, select(changes, task='pooled_distractor', direction=direction, control=null), LABELS[direction], color)
    ax.axhline(0, color='gray', linestyle=':')
    ax.set(title=f'Change from 5k: vs {null}', ylabel='Paired effect change (raw ΔLD)')
    ax.legend()
save(fig, '07_changes_from_5k')

## 6. 探索性 head maps 与覆盖率

下面展示全部 heads 的 pooled distractor 均值，统一色标。它用于发现分布变化，不提供额外“显著 head”的判定；不能从测试集热图重新选择 head 后再宣称独立验证。预先冻结的额外候选结果见 `exploratory_heads.csv`。

固定 cohort 覆盖 test 的 40%。多个例子共享提示或词汇，prompt-cluster bootstrap 只处理重复提示，不解决共享词汇、有限模板和单训练 seed 的全部依赖。

In [ ]:
maps = load('head_maps.csv')
steps = sorted({int(r['step']) for r in maps})
limit = max(abs(float(r['mean'])) for r in maps)
fig, axes = plt.subplots(2, 6, figsize=(16, 6), layout='constrained')
for i, direction in enumerate(CROSS):
    for j, step in enumerate(steps):
        matrix = np.zeros((12, 8))
        for r in select(maps, step=step, direction=direction):
            matrix[int(r['layer']), int(r['head'])] = float(r['mean'])
        im = axes[i,j].imshow(matrix, vmin=-limit, vmax=limit, cmap='RdBu_r', aspect='auto', origin='lower')
        axes[i,j].scatter([3], [8], marker='s', s=65, facecolors='none', edgecolors='black')
        axes[i,j].set(title=f'{step/1000:g}k', xlabel='Head', xticks=[0,3,7])
        if j == 0: axes[i,j].set_ylabel(LABELS[direction] + '\nLayer')
fig.colorbar(im, ax=axes, label='Clean-minus-opposite-number-shuffled effect (raw ΔLD)', shrink=.8)
save(fig, '08_exploratory_head_maps')
fig, ax = plt.subplots(figsize=(7, 4), layout='constrained')
ax.bar([TASK_LABELS[r['task']] for r in coverage], [float(r['coverage'])*100 for r in coverage], color='#477797')
ax.set(ylabel='Test pairs retained (%)', ylim=(0,100), title='Fixed-cohort coverage')
for i,r in enumerate(coverage):
    ax.text(i, float(r['coverage'])*100+2, f"{r['patched_pairs']}/{r['test_pairs']}", ha='center')
save(fig, '09_cohort_coverage')

## 历史 confirmatory 统计与复现

保留原始 best-checkpoint 统计，不将其与新的跨 checkpoint 探索性区间混为一谈。历史 Holm 校正范围是每个 site、direction、null 内的三个 distractor tasks，不是所有 heads 或所有 checkpoint。以下 `0` 尾部计数显示为“未抽到反向尾部”，不解释为真实 p=0；严格推断需要额外校准，而不能提高显示精度。

报告入口：`reports/cross_language_sva/report_zh.md`。新增消融和联合 patching 尚未实施。

In [ ]:
historical = load('historical_confirmatory.csv')
rows = []
for r in historical:
    if r['direction'] not in CROSS or r['task'] == 'simple': continue
    p = r['holm_p']
    if p and float(p) == 0:
        p = 'No opposite-tail draws / 10,000 (approx.)'
    elif p:
        p = f'{float(p):.4f}'
    else:
        p = 'Not adjusted (pooled)'
    rows.append({'direction': r['direction'], 'null': r['control'], 'task': r['task'],
                 'effect': f"{float(r['mean_difference']):+.4f}", 'historical Holm': p})
table(rows, list(rows[0]))
print('Exported', len(list(FIGURES.glob('*.png'))), 'PNG figures.')